# 00_config / 02__favorites

Manages the `main.config.favorites` table — tickers that are **always ingested**
regardless of index membership.

**How to use:**
- Run the whole notebook once to create the table
- Use the ADD / REMOVE cells below to manage your list
- Re-run `03__tickers_master` after any changes to rebuild the universe

In [0]:
%run "./01__tickers"

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.config.favorites (
        ticker   STRING NOT NULL,
        company  STRING,
        note     STRING,
        added_at TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

print(f"✓ {CATALOG}.config.favorites ready")
spark.sql(f"SELECT * FROM {CATALOG}.config.favorites ORDER BY ticker").show()

## Add favorites
Edit the list and run the cell. Safe to re-run — uses MERGE so no duplicates.

In [0]:
from pyspark.sql import Row
from datetime import datetime

# ── Edit this list ────────────────────────────────────────────────────────────
ADD_TICKERS = [
    # {"ticker": "TSM",  "company": "Taiwan Semiconductor", "note": ""},
    # {"ticker": "ASML", "company": "ASML Holding",         "note": ""},
]
# ─────────────────────────────────────────────────────────────────────────────

if ADD_TICKERS:
    rows = [Row(ticker=r["ticker"], company=r.get("company"), note=r.get("note", ""), added_at=datetime.now()) for r in ADD_TICKERS]
    spark.createDataFrame(rows).createOrReplaceTempView("_favorites_to_add")
    spark.sql(f"""
        MERGE INTO {CATALOG}.config.favorites AS target
        USING _favorites_to_add AS source
        ON target.ticker = source.ticker
        WHEN NOT MATCHED THEN
            INSERT (ticker, company, note, added_at)
            VALUES (source.ticker, source.company, source.note, source.added_at)
    """)
    print(f"✓ Added {len(ADD_TICKERS)} ticker(s)")
else:
    print("No tickers to add — edit ADD_TICKERS above.")

spark.sql(f"SELECT * FROM {CATALOG}.config.favorites ORDER BY ticker").show()

## Remove favorites

In [0]:
# ── Edit this list ────────────────────────────────────────────────────────────
REMOVE_TICKERS = [
    # "BABA",
]
# ─────────────────────────────────────────────────────────────────────────────

if REMOVE_TICKERS:
    tickers_sql = ", ".join(f"'{t}'" for t in REMOVE_TICKERS)
    spark.sql(f"DELETE FROM {CATALOG}.config.favorites WHERE ticker IN ({tickers_sql})")
    print(f"✓ Removed {len(REMOVE_TICKERS)} ticker(s)")
else:
    print("No tickers to remove — edit REMOVE_TICKERS above.")

spark.sql(f"SELECT * FROM {CATALOG}.config.favorites ORDER BY ticker").show()

## Current favorites

In [0]:
spark.sql(f"""
    SELECT ticker, company, note, added_at,
           DATEDIFF(day, added_at, current_timestamp()) AS days_tracked
    FROM {CATALOG}.config.favorites
    ORDER BY added_at DESC
""").show(truncate=False)